In [10]:
import json

jsonl_file = "/qumulo/shared_data/aofei_summer/RegTok/data/RegAlign_GPT5_mini_v12.jsonl"
with open(jsonl_file, "r") as f:
    data = [json.loads(line) for line in f]

In [13]:
data[1400], len(data)

({'image_id': 1400,
  'dialogues': [{'User': "Let's review this breast ultrasound together. I'll ask you to identify and mark important regions.",
    'Assistant': "Understood. I'm ready to review the ultrasound and provide segmentation and localization."},
   {'User': 'Please segment the lesion visible in this image.',
    'Assistant': 'Segmented region corresponding to a benign breast tumor [M2_11].',
    'mask_ids_order': [0]},
   {'User': 'Provide the diagnosis and mark the affected area.',
    'Assistant': 'Findings consistent with a benign breast tumor, localized and segmented as [M2_11].',
    'mask_ids_order': [0]}],
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train/benign (343)_ultrasound_breast.png',
  'mask_code': [[542,
    'M2_11',
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/train_mask/benign (343)_ultrasound_breast_benign+tumor.png']]},
 12508)

In [14]:
# preprocess to MLLM training style
"""
target format:
{
    "id": id,
    "image": image_path,
    "conversations": [
    {
        "from":  "human", "value": user_message
    },
    {
        "from":  "assistant", "value": assistant_message
    }, ...
    ],
    "mask_files": {
        "mask_id": mask_file_path,
    },
    "mask_orders": [] # the ids of mentioned masks one by one, separated by commas
}
"""

def preprocess_data(item):
    """
    Return two items: caption_item and conv_item in the target MLLM format.
    This implementation handles the example input shape where:
    - item['image_file'] or item['image'] holds image path
    - item['caption'] may be a dict with 'text' and 'mask_ids_order'
    - item['dialogues'] is a list of dicts with 'User', 'Assistant', and 'mask_ids_order'
    - item['mask_code'] is a list of [id, label, path]
    """
    # id and image
    id_ = item['image_id']
    image = item.get('image_file') or item.get('image') or item.get('image_path') or item.get('img') or ''

    # Build mask_files from several possible formats
    mask_files = {}
    mask_codes = {}
    mask_ids_list = []
    if 'mask_code' in item and isinstance(item['mask_code'], list):
        for entry in item['mask_code']:
            # expected entry: [id, label, path]
            try:
                mid = int(entry[0])
                code = entry[1] if len(entry) > 1 else None
                path = entry[2] if len(entry) > 2 else None
                if path:
                    mask_files[str(mid)] = path
                    mask_codes[str(mid)] = code
                    mask_ids_list.append(mid)
            except Exception:
                continue
    else:
        mf = item.get('mask_files') or item.get('masks') or {}
        if isinstance(mf, list):
            mask_files = {str(i): p for i, p in enumerate(mf)}
        elif isinstance(mf, dict):
            # convert keys to strings
            mask_files = {str(k): v for k, v in mf.items()}

# Full conversations: normalize dialogues to the target format
    convs = []
    mask_orders_conv = []
    # If 'dialogues' exists (example uses this), iterate and convert
    if 'dialogues' in item and isinstance(item['dialogues'], list):
        for turn in item['dialogues']:
            if not isinstance(turn, dict):
                continue
            user = turn.get('User') or turn.get('user') or turn.get('question') or turn.get('prompt')
            assistant = turn.get('Assistant') or turn.get('assistant') or turn.get('answer') or turn.get('response')
            if user is not None:
                convs.append({'from': 'human', 'value': user})
            if assistant is not None:
                convs.append({'from': 'assistant', 'value': assistant})
            # extend global mask_orders with any ids mentioned in this turn (preserve order, uniqueness)
            if 'mask_ids_order' in turn and isinstance(turn['mask_ids_order'], (list, tuple)):
                for mid in turn['mask_ids_order']:
                    if mid > 100:
                        mask_orders_conv.append(mid)
                    else:
                        if mid < len(mask_ids_list):
                            mask_orders_conv.append(mask_ids_list[mid])
                        else:
                            continue
    else:
        # Fallback: try single-turn keys
        q = item.get('question') or item.get('user') or item.get('prompt')
        a = item.get('answer') or item.get('response') or item.get('assistant') or item.get('caption')
        if q is not None:
            convs.append({'from': 'human', 'value': q})
        if a is not None:
            convs.append({'from': 'assistant', 'value': a})

    conv_item = {
        'id': id_,
        'image': image,
        'conversations': convs,
        'mask_files': mask_files,
        'mask_codes': mask_codes,
        'mask_orders': mask_orders_conv,
    }

    return conv_item

# Example usage with your provided input (uncomment to test):
# example = { ... }
# caption_item, conv_item = preprocess_data(example)
# print(caption_item)
# print(conv_item)

In [15]:
# ...existing code...
import re
def check_conversation(item):
    conversations = item.get("conversations", [])
    text = " ".join(conv.get("value", "") for conv in conversations)
    # pattern captures modality_index and code_index from tokens like [M2_15]
    pattern = r'\[M(\d+)_(\d+)\]'
    matches = re.findall(pattern, text)  # returns list of (modality_index, code_index) tuples
    flag = len(matches) == len(item.get("mask_orders", []))
    
    if not flag:
        mask_codes = item.get("mask_codes", {})
        mask_order_ids = item.get("mask_orders", [])
        # print(mask_codes, mask_order_ids)
        if len(item.get("mask_orders", [])) > len(matches):
            # print("Mask Order IDs:", mask_order_ids)
            matched_Codes = [f"M{i[0]}_{i[1]}" for i in matches]
            id_to_Codes = [mask_codes.get(str(i), "") for i in mask_order_ids]
            # print(matched_Codes, id_to_Codes)
            if matched_Codes == id_to_Codes[:len(matches)]:
                flag = True
                print("Matched Codes:", matched_Codes, id_to_Codes)
                item['mask_orders'] = mask_order_ids[:len(matches)]
            # elif len(mask_codes) == len(set([i[1] for i in list(mask_codes.items())])):
            #     print("Mask Codes:", mask_codes)
            # else:
            #     print("fAILED Mask Codes:", mask_codes)

    return flag

In [26]:
correct_items_conversation = []
failed_items_conversation = []
failed_image_id = []
for i in data:
    processed_i = preprocess_data(i)
    # processed_i['conversations'] = processed_i['conversations'][2:]
    if check_conversation(processed_i):
        correct_items_conversation.append(processed_i)
    else:
        failed_image_id.append(i['image_id'])
        failed_items_conversation.append(processed_i)
    

Matched Codes: ['M0_16', 'M0_29', 'M0_13', 'M0_1', 'M0_28', 'M0_10'] ['M0_16', 'M0_29', 'M0_13', 'M0_1', 'M0_28', 'M0_10', 'M0_10', 'M0_10']
Matched Codes: ['M2_11'] ['M2_11', 'M2_11']
Matched Codes: ['M2_11'] ['M2_11', 'M2_11']
Matched Codes: ['M2_11'] ['M2_11', 'M2_11']
Matched Codes: ['M2_22'] ['M2_22', 'M2_22']
Matched Codes: ['M2_22'] ['M2_22', 'M2_22']
Matched Codes: ['M2_22'] ['M2_22', 'M2_22']
Matched Codes: ['M2_11'] ['M2_11', 'M2_11']
Matched Codes: ['M2_11'] ['M2_11', 'M2_11']
Matched Codes: ['M2_11', 'M2_11', 'M2_11'] ['M2_11', 'M2_11', 'M2_11', 'M2_11']
Matched Codes: ['M2_11', 'M2_11', 'M2_11'] ['M2_11', 'M2_11', 'M2_11', 'M2_11']
Matched Codes: ['M2_11', 'M2_11', 'M2_11'] ['M2_11', 'M2_11', 'M2_11', 'M2_11']
Matched Codes: ['M2_22', 'M2_22', 'M2_22'] ['M2_22', 'M2_22', 'M2_22', 'M2_22']
Matched Codes: ['M2_11', 'M2_11', 'M2_11'] ['M2_11', 'M2_11', 'M2_11', 'M2_11']
Matched Codes: ['M2_22', 'M2_22'] ['M2_22', 'M2_22', 'M2_22', 'M2_22']
Matched Codes: ['M2_22'] ['M2_22', '

In [38]:
len(correct_items_conversation), len(failed_items_conversation)

(12445, 63)

In [39]:
correct_items_conversation[0]

{'id': 0,
 'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train/amos_0532_6_MRI_abdomen.png',
 'conversations': [{'from': 'human',
   'value': "Let's review this abdominal MRI together. I'll ask you to identify and segment key structures."},
  {'from': 'assistant',
   'value': "Understood — I'm ready to identify and segment relevant structures."},
  {'from': 'human', 'value': 'Please segment both kidneys in this MRI.'},
  {'from': 'assistant',
   'value': 'Left kidney segmented as [M1_11]; right kidney segmented as [M1_3].'},
  {'from': 'human',
   'value': 'Now identify and segment the major abdominal vessels visible here.'},
  {'from': 'assistant',
   'value': 'Posterior vena cava (postcava) segmented as [M1_9]; aorta segmented as [M1_18].'}],
 'mask_files': {'4701': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/MRI/train_mask/amos_0532_6_MRI_abdomen_right+kidney.png',
  '4702': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos2

In [51]:
safe_data = []
unsafe_data = []
for i in correct_items_conversation:
    if len(i['mask_orders']) <= 5:
        safe_data.append(i)
    else:
        unsafe_data.append(i)

In [52]:
unsafe_data[90], len(unsafe_data)

({'id': 745,
  'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train/amos_0358_192_CT_abdomen.png',
  'conversations': [{'from': 'human',
    'value': "Let's examine this abdominal CT. Be prepared to segment renal and vascular structures."},
   {'from': 'assistant',
    'value': 'Ready to segment renal and vascular structures on the CT.'},
   {'from': 'human',
    'value': 'Segment the right and left kidneys and the renal region.'},
   {'from': 'assistant',
    'value': 'Right kidney segmented as [M0_19]; left kidney segmented as [M0_26]; renal region segmented as [M0_21].'},
   {'from': 'human', 'value': 'Localize the aorta, postcava, and pancreas.'},
   {'from': 'assistant',
    'value': 'Aorta localized as [M0_1]; postcava localized as [M0_28]; pancreas segmented as [M0_13].'}],
  'mask_files': {'86916': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train_mask/amos_0358_192_CT_abdomen_aorta.png',
   '86936': '/qumulo/shared_data/aof

In [53]:
len(safe_data)

12347

In [46]:
max([len(i['mask_orders']) for i in unsafe_data])

10

In [54]:
# save this conversation item
# items_to_save = [i for i in correct_items_conversation]
items_to_save = [i for i in safe_data]
print(len(items_to_save))
save_path = "/qumulo/shared_data/aofei_summer/data/RegAlign/Seg_instruct_12k_1104_safe.json"
# # items_to_save[0]
with open(save_path, "w") as f:
    json.dump(items_to_save, f)

12347


In [1]:
# load save_path
import json
save_path = "/qumulo/shared_data/aofei_summer/data/RegAlign/Seg_instruct_12k_1104_safe.json"
with open(save_path, "r") as f:
    items_to_save = json.load(f)
len(items_to_save)

12347

In [70]:
len(items_to_save)
no_ins_items_to_save = []
for item in items_to_save:
    new_item = item.copy()
    new_item['conversations'] = new_item['conversations'][2:]
    no_ins_items_to_save.append(new_item)

In [73]:
# no_ins_items_to_save[2]
# save it
save_path_no_ins = "/qumulo/shared_data/aofei_summer/data/RegAlign/Seg_instruct_12k_1104_no_ins.json"
with open(save_path_no_ins, "w") as f:
    json.dump(no_ins_items_to_save, f)

In [19]:
old_instruct = "/qumulo/shared_data/aofei_summer/data/LVLM/HuatuoGPT/60k_PubMedVision_InstructionTuning_VQA.json"
# old_instruct = "/qumulo/shared_data/aofei_summer/data/LVLM/HuatuoGPT/300k_PubMedVision_InstructionTuning_VQA.json"
import json
with open(old_instruct, "r") as f:
    old_data = json.load(f)
for i in old_data:
    i['mask_files'] = None
    i['mask_codes'] = None
    i['mask_orders'] = []
len(old_data)

60000

In [20]:
safe_full = []
large_full = []
for item in old_data:
    if len(item['image']) <=4:
        safe_full.append(item)
    else:
        large_full.append(item)
print(len(safe_full), len(large_full))
max([len(item['image']) for item in large_full])


58772 1228


19

In [21]:
new_items_to_save = safe_full + items_to_save
len(new_items_to_save)

71119

In [22]:
len(new_items_to_save), new_items_to_save[0], new_items_to_save[-1]

(71119,
 {'image': ['images_all/pmc_81861_0.jpg'],
  'conversations': [{'from': 'human',
    'value': 'What is the subcellular localization of the ABHD17A protein in COS-7 cells based on this image?'},
   {'from': 'gpt',
    'value': 'According to the image, the ABHD17A protein is localized to the plasma membrane and various endosomal compartments, including early endosomes (Rab5), late endosomes (Rab7), and recycling endosomes (Rab11). The protein is also found in the Golgi apparatus, as indicated by the GM130 marker. The image also shows the localization of a mutant form of ABHD17A (ABHD17A ΔN) in relation to the Golgi apparatus. Additionally, the image demonstrates the co-expression of wild-type and mutant forms of mCherry-tagged ABHD17A with EGFP-N-Ras in COS-7 cells.'}],
  'id': 'Instruction-Tuning_81861',
  'modality': 'Microscopy Images',
  'body_part': 'Cell',
  'mask_files': None,
  'mask_codes': None,
  'mask_orders': []},
 {'id': 12507,
  'image': '/qumulo/shared_data/aofei_

In [23]:
data_path = "/qumulo/shared_data/aofei_summer/data/RegAlign/final_data/Instruct_71k_1104.json"
with open(data_path, "w") as f:
    json.dump(new_items_to_save, f)

In [68]:
new_items_to_save[0]

{'image': ['images_all/pmc_81861_0.jpg'],
 'conversations': [{'from': 'human',
   'value': 'What is the subcellular localization of the ABHD17A protein in COS-7 cells based on this image?'},
  {'from': 'gpt',
   'value': 'According to the image, the ABHD17A protein is localized to the plasma membrane and various endosomal compartments, including early endosomes (Rab5), late endosomes (Rab7), and recycling endosomes (Rab11). The protein is also found in the Golgi apparatus, as indicated by the GM130 marker. The image also shows the localization of a mutant form of ABHD17A (ABHD17A ΔN) in relation to the Golgi apparatus. Additionally, the image demonstrates the co-expression of wild-type and mutant forms of mCherry-tagged ABHD17A with EGFP-N-Ras in COS-7 cells.'}],
 'id': 'Instruction-Tuning_81861',
 'modality': 'Microscopy Images',
 'body_part': 'Cell',
 'mask_files': None,
 'mask_codes': None,
 'mask_orders': []}

In [2]:
full_inst_data_path = "/qumulo/shared_data/aofei_summer/data/LVLM/HuatuoGPT/300k_PubMedVision_InstructionTuning_VQA.json"
with open(full_inst_data_path, "r") as f:
    full_inst_data = json.load(f)

In [3]:
full_inst_data[0]

{'image': ['images_all/pmc_81861_0.jpg'],
 'conversations': [{'from': 'human',
   'value': 'What is the subcellular localization of the ABHD17A protein in COS-7 cells based on this image?'},
  {'from': 'gpt',
   'value': 'According to the image, the ABHD17A protein is localized to the plasma membrane and various endosomal compartments, including early endosomes (Rab5), late endosomes (Rab7), and recycling endosomes (Rab11). The protein is also found in the Golgi apparatus, as indicated by the GM130 marker. The image also shows the localization of a mutant form of ABHD17A (ABHD17A ΔN) in relation to the Golgi apparatus. Additionally, the image demonstrates the co-expression of wild-type and mutant forms of mCherry-tagged ABHD17A with EGFP-N-Ras in COS-7 cells.'}],
 'id': 'Instruction-Tuning_81861',
 'modality': 'Microscopy Images',
 'body_part': 'Cell'}

In [13]:
safe_full = []
large_full = []
for item in full_inst_data:
    if len(item['image']) <=4:
        safe_full.append(item)
    else:
        large_full.append(item)
len(safe_full), len(large_full)

(293635, 6365)

In [14]:
max([len(item['image']) for item in large_full])

29

In [16]:
len(full_inst_data), len(items_to_save)

(300000, 12347)

In [17]:
full_inst_seg_data = items_to_save + safe_full
len(full_inst_seg_data)

305982

In [67]:
full_inst_seg_data[10000]

{'id': 10150,
 'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/NeoPolyp/NeoPolyp/train/367f150f68fa7ee0ca748f990aeafa35_endoscope_colon.png',
 'conversations': [{'from': 'human',
   'value': "Let's examine this colonoscopy image together. I'll ask you to identify and segment any polyps you see."},
  {'from': 'assistant',
   'value': "Okay — I'm ready to review the endoscopic view and provide segmentation."},
  {'from': 'human',
   'value': 'Please identify and segment the polyp and indicate if it is neoplastic or not.'},
  {'from': 'assistant',
   'value': 'A non-neoplastic polyp is identified and segmented as [M15_7]. The same polyp region is also represented as a general polyp segmentation [M15_29].'}],
 'mask_files': {'422': '/qumulo/shared_data/aofei_summer/data/BiomedParse/NeoPolyp/NeoPolyp/train_mask/367f150f68fa7ee0ca748f990aeafa35_endoscope_colon_non-neoplastic+polyp.png',
  '423': '/qumulo/shared_data/aofei_summer/data/BiomedParse/NeoPolyp/NeoPolyp/train_mask/367f1

In [18]:
data_path = "/qumulo/shared_data/aofei_summer/data/RegAlign/final_data/Instruct_306k_1104.json"
with open(data_path, "w") as f:
    json.dump(full_inst_seg_data, f)